Auth to Huggingface

In [1]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BitsAndBytes Config

In [2]:
from transformers import BitsAndBytesConfig
import torch

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

Output Directory

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install Dependencies

In [4]:
!pip install -q -U torch transformers datasets accelerate bitsandbytes peft trl

Begin LoRA Fine-Tuning replecating Esposito, Gabriele. LLMs in the SIEM Loop: A Contract-Based Framework for Threat Detection with an Evaluation on Windows Telemetry and MITRE ATT&CK Mapping

In [8]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from huggingface_hub import notebook_login

# 1. Authenticate with Hugging Face (Required to download Mistral)
print("Please log in to Hugging Face:")
notebook_login()

# 2. Load the Dataset
print("Loading dataset...")
ds = load_dataset("tumeteor/Security-TTP-Mapping")

# 3. Configure 4-bit Quantization (NF4)
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. Load Model and Tokenizer
model_id = "mistralai/Mistral-7B-Instruct-v0.1" # Standard base for this task

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fixes issue with fp16 training

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=nf4_config,
    device_map="auto",
    trust_remote_code=True
)

# 5. Prepare Model for LoRA
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# The missing PEFT Config from the paper
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 6. Formatting Function (from paper)
def create_prompt(example):
    bos_token = "<s>"
    eos_token = "</s>"
    system_message = (
        "You are a cybersecurity assistant. "
        "Your task is to analyze a system log and assign the most appropriate MITRE ATT&CK technique(s)."
    )
    log_text = example["text1"]
    mitre_labels = "".join(example["labels"]).strip()

    full_prompt = (
        f"{bos_token}\n"
        f"### Instruction:\n{system_message}\n\n"
        f"### Log:\n{log_text}\n\n"
        f"### Response:\n{mitre_labels}\n"
        f"{eos_token}"
    )
    return full_prompt

from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

# 7. Training Arguments
args = SFTConfig(
    output_dir="/content/drive/MyDrive/CSC699/HF/siem-finetuned",
    max_length=1024,
    max_steps=2400,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=5,
    learning_rate=1e-4,
    bf16=True,
    lr_scheduler_type="constant",
    metric_for_best_model="eval_loss",
    report_to=[]
)

# 8. Initialize Trainer (Removed peft_config since the model is already wrapped!)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    formatting_func=create_prompt,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 9. Execute Training
print("Starting training...")
trainer.train()

# 10. Save the final model adapter
trainer.model.save_pretrained("/content/drive/MyDrive/CSC699/HF/final_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/CSC699/HF/final_adapter")
print("Training complete and adapter saved to your CSC699/HF folder!")

Please log in to Hugging Face:
Loading dataset...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/2630 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


TypeError: new(): invalid data type 'str'

In [9]:
from transformers import EarlyStoppingCallback

# 1. Rename the conflicting column across the entire dataset
ds = ds.rename_column("labels", "mitre_tags")

# 2. Update the formatting function to use the new column name
def create_prompt(example):
    bos_token = "<s>"
    eos_token = "</s>"
    system_message = (
        "You are a cybersecurity assistant. "
        "Your task is to analyze a system log and assign the most appropriate MITRE ATT&CK technique(s)."
    )
    log_text = example["text1"]

    # Grab the string from our renamed column!
    mitre_tags = "".join(example["mitre_tags"]).strip()

    full_prompt = (
        f"{bos_token}\n"
        f"### Instruction:\n{system_message}\n\n"
        f"### Log:\n{log_text}\n\n"
        f"### Response:\n{mitre_tags}\n"
        f"{eos_token}"
    )
    return full_prompt

# 3. Re-initialize the Trainer with the patched dataset
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    formatting_func=create_prompt,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 4. Execute Training
print("Starting training...")
trainer.train()

# 5. Save the final model adapter
trainer.model.save_pretrained("/content/drive/MyDrive/CSC699/HF/final_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/CSC699/HF/final_adapter")
print("Training complete and adapter saved to your CSC699/HF folder!")

Applying formatting function to train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14936 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/2630 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/2630 [00:00<?, ? examples/s]

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Starting training...


Step,Training Loss,Validation Loss
200,1.001980,0.881543
400,0.960285,0.836336
600,0.859996,0.817610
800,0.811482,0.795071
1000,0.673048,0.801774
1200,0.688368,0.786018
1400,0.676030,0.768188
1600,0.494946,0.796352
1800,0.510817,0.787619


Training complete and adapter saved to your CSC699/HF folder!
